# Image Anomaly Detection with MNIST

This notebook demonstrates quantum anomaly detection on image data using MNIST digits.
We treat one digit class as "normal" and another as "anomaly", then compare three
quantum methods against classical baselines.

**Setup**: Digit 0 = normal, Digit 1 = anomaly (since they look structurally different).

**Quantum methods**: Quantum Kernel + One-Class SVM, VQC Autoencoder, Quantum Distance k-NN.

**Classical baselines**: Isolation Forest, One-Class SVM, LOF, Elliptic Envelope.

In [ ]:
# ============================================================
# Configuration — change these values to experiment
# ============================================================
SEED = 42
N_QUBITS = 8
NORMAL_DIGIT = 0
ANOMALY_DIGITS = (1,)
N_NORMAL = 400
N_ANOMALY = 40
CONTAMINATION = 0.05
N_KERNEL_SAMPLES = 100
CIRCUIT_SCALE = 0.2  # Scale factor for circuit drawings


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from quantum_anomaly_detection.visualization.plots import (
    plot_optimization_history,
    plot_2d_scatter,
    plot_roc_curves,
    plot_kernel_matrix,
)
from quantum_anomaly_detection.evaluation.metrics import compute_metrics, format_results_table


## 1. Data Loading & Exploration

We load MNIST from OpenML and set up an anomaly detection problem where one digit
class is considered normal and another is the anomaly.

In [ ]:
from quantum_anomaly_detection.data.image import load_mnist_anomaly, preprocess_images

X_raw, y = load_mnist_anomaly(
    normal_digit=NORMAL_DIGIT, anomaly_digits=ANOMALY_DIGITS,
    n_normal=N_NORMAL, n_anomaly=N_ANOMALY, seed=SEED
)
print(f'Dataset: {len(X_raw)} images ({(y==0).sum()} normal, {(y==1).sum()} anomaly)')
print(f'Image shape: {X_raw.shape[1]} pixels (28x28 flattened)')

In [ ]:
# Show sample images
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
normal_idx = np.where(y == 0)[0][:8]
anomaly_idx = np.where(y == 1)[0][:8]

for i, idx in enumerate(normal_idx):
    axes[0, i].imshow(X_raw[idx].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    if i == 0: axes[0, i].set_ylabel('Normal', fontsize=12)

for i, idx in enumerate(anomaly_idx):
    axes[1, i].imshow(X_raw[idx].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')
    if i == 0: axes[1, i].set_ylabel('Anomaly', fontsize=12)

fig.suptitle(f'MNIST: Normal=digit {NORMAL_DIGIT}, Anomaly=digit {ANOMALY_DIGITS}')
plt.tight_layout()
plt.show()

## 2. Preprocessing

MNIST images have 784 pixels. We reduce this to N_QUBITS dimensions using PCA,
then scale to [0, pi] for quantum feature map encoding. This dramatic dimensionality
reduction is necessary for quantum processing but retains the most important variance.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=SEED, stratify=y
)

X_train = preprocess_images(X_train_raw, n_components=N_QUBITS, fit_data=X_train_raw)
X_test = preprocess_images(X_test_raw, n_components=N_QUBITS, fit_data=X_train_raw)

X_train_normal = X_train[y_train == 0]
print(f'Train: {len(X_train)} ({(y_train==0).sum()} normal), Test: {len(X_test)}')
print(f'PCA features: {X_train.shape[1]} components, range [{X_train.min():.2f}, {X_train.max():.2f}]')

## 3. Quantum Kernel + One-Class SVM

The quantum kernel computes the overlap |<phi(x_i)|phi(x_j)>|^2 between quantum states
encoding the PCA features of each image. The ZZ feature map creates entanglement between
qubit pairs, allowing the kernel to capture non-linear correlations between PCA components
that a classical RBF kernel might miss.

In [ ]:
from quantum_anomaly_detection.circuits.feature_maps import build_zz_feature_map
from quantum_anomaly_detection.methods.quantum_kernel import quantum_kernel_svm, compute_kernel_matrix
from quantum_anomaly_detection.visualization.plots import plot_kernel_matrix

feature_map_zz = build_zz_feature_map(N_QUBITS, reps=1)
print(f'ZZ Feature Map: {N_QUBITS} qubits, {feature_map_zz.num_parameters} parameters')
feature_map_zz.draw('mpl', fold=20, scale=CIRCUIT_SCALE)

In [ ]:
rng = np.random.default_rng(SEED)
n_sub = min(N_KERNEL_SAMPLES, len(X_train_normal))
idx_train = rng.choice(len(X_train_normal), n_sub, replace=False)
idx_test = rng.choice(len(X_test), min(N_KERNEL_SAMPLES, len(X_test)), replace=False)

preds_qk, scores_qk, _ = quantum_kernel_svm(
    X_train_normal[idx_train], X_test[idx_test], feature_map_zz, nu=CONTAMINATION
)

# Kernel matrix heatmap
K = compute_kernel_matrix(X_train_normal[idx_train[:25]], feature_map_zz)
plot_kernel_matrix(K, title='Quantum Kernel Matrix (MNIST PCA Features)')
plt.show()

In [ ]:
from quantum_anomaly_detection.evaluation.metrics import compute_metrics

results = {}
m = compute_metrics(y_test[idx_test], preds_qk, scores_qk)
results['Quantum Kernel SVM'] = m
print('Quantum Kernel One-Class SVM:', {k: f'{v:.3f}' for k, v in m.items()})

## 4. VQC Autoencoder

The quantum autoencoder compresses the PCA-encoded image features into a latent space
of N_QUBITS//2 qubits. Images that are structurally similar to the training set (normal
digit) should compress well, while anomaly digits will have higher reconstruction error.

In [ ]:
from quantum_anomaly_detection.circuits.autoencoder import build_autoencoder_compact

n_latent = N_QUBITS // 2
ae_compact, ae_encoder, ae_decoder = build_autoencoder_compact(
    N_QUBITS, n_latent, encoder_reps=1, decoder_reps=1
)
print(f'Autoencoder: {N_QUBITS} qubits -> {n_latent} latent')
print('\nCompact view:')
display(ae_compact.draw('mpl', scale=CIRCUIT_SCALE))
print('\nEncoder detail:')
display(ae_encoder.draw('mpl', fold=20, scale=CIRCUIT_SCALE))


In [ ]:
from quantum_anomaly_detection.methods.vqc_autoencoder import train_autoencoder, score_anomalies, detect_anomalies

# Train on normal images (small subset for speed)
train_sub = X_train_normal[:50]
params_ae, circuit_ae, history_ae = train_autoencoder(
    train_sub, n_latent=n_latent, encoder_reps=1, decoder_reps=1,
    maxiter=100, seed=SEED
)

plot_optimization_history(history_ae, title='VQC Autoencoder Training (MNIST)')
plt.show()

In [ ]:
# Score all test images
scores_ae = score_anomalies(X_test, params_ae, circuit_ae, n_latent)
preds_ae = detect_anomalies(scores_ae, contamination=CONTAMINATION)

m = compute_metrics(y_test, preds_ae, scores_ae)
results['VQC Autoencoder'] = m
print('VQC Autoencoder:', {k: f'{v:.3f}' for k, v in m.items()})

In [ ]:
# Show highest-error images (most anomalous according to autoencoder)
top_anomaly_idx = np.argsort(scores_ae)[-8:]
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, idx in enumerate(top_anomaly_idx):
    axes[i].imshow(X_test_raw[idx].reshape(28, 28), cmap='gray')
    label = 'A' if y_test[idx] == 1 else 'N'
    axes[i].set_title(f'{label} ({scores_ae[idx]:.2f})', fontsize=8)
    axes[i].axis('off')
fig.suptitle('Highest Reconstruction Error (A=anomaly, N=normal)')
plt.tight_layout()
plt.show()

## 5. Quantum Distance k-NN

Quantum distance estimation computes pairwise distances using the fidelity between
quantum states: d(x_1, x_2) = sqrt(1 - F). Images far from their k nearest neighbors
in quantum feature space are flagged as anomalies.

In [ ]:
from quantum_anomaly_detection.circuits.feature_maps import build_angle_encoding_map
from quantum_anomaly_detection.methods.quantum_distance import detect_anomalies_knn

fm_angle = build_angle_encoding_map(N_QUBITS)
fm_angle.draw('mpl', fold=20, scale=CIRCUIT_SCALE)

In [ ]:
# Distance matrix on subset
n_dist = min(80, len(X_test))
idx_dist = rng.choice(len(X_test), n_dist, replace=False)

preds_qdist, scores_qdist = detect_anomalies_knn(
    X_test[idx_dist], fm_angle, k=5, contamination=CONTAMINATION
)

m = compute_metrics(y_test[idx_dist], preds_qdist, scores_qdist)
results['Quantum Distance k-NN'] = m
print('Quantum Distance k-NN:', {k: f'{v:.3f}' for k, v in m.items()})

## 6. Classical Benchmarks (Level A)

In [ ]:
from quantum_anomaly_detection.classical.benchmarks import (
    run_isolation_forest, run_svm, run_lof, run_elliptic_envelope
)

preds_if, scores_if = run_isolation_forest(X_train_normal, X_test, CONTAMINATION, SEED)
results['Isolation Forest'] = compute_metrics(y_test, preds_if, scores_if)

preds_ocsvm, scores_ocsvm = run_svm(X_train_normal, X_test, nu=CONTAMINATION)
results['One-Class SVM (RBF)'] = compute_metrics(y_test, preds_ocsvm, scores_ocsvm)

preds_lof, scores_lof = run_lof(X_train_normal, X_test, contamination=CONTAMINATION)
results['LOF'] = compute_metrics(y_test, preds_lof, scores_lof)

preds_ee, scores_ee = run_elliptic_envelope(X_train_normal, X_test, CONTAMINATION)
results['Elliptic Envelope'] = compute_metrics(y_test, preds_ee, scores_ee)

print('Classical methods computed.')

## 7. Comparison

### Results Table

In [ ]:
from quantum_anomaly_detection.evaluation.metrics import format_results_table

df = format_results_table(results)
print(df.to_string())

In [ ]:
from quantum_anomaly_detection.visualization.plots import plot_roc_curves

roc_data = {
    'Quantum Kernel': (y_test[idx_test], scores_qk),
    'VQC Autoencoder': (y_test, scores_ae),
    'Quantum Distance': (y_test[idx_dist], scores_qdist),
    'Isolation Forest': (y_test, scores_if),
    'One-Class SVM (RBF)': (y_test, scores_ocsvm),
    'LOF': (y_test, scores_lof),
}

plot_roc_curves(roc_data, title='ROC Curves — MNIST Anomaly Detection')
plt.show()

In [ ]:
from quantum_anomaly_detection.visualization.plots import plot_2d_scatter

plot_2d_scatter(X_test, y_test, title='True Labels (PCA projection)', method_name='Ground Truth')
plt.show()

plot_2d_scatter(X_test, preds_ae, title='VQC Autoencoder Predictions', method_name='VQC AE')
plt.show()

## Discussion

- **Quantum Kernel One-Class SVM** can capture complex feature correlations through the ZZ
  entanglement structure, but with only 8 PCA components, much of the image information
  is lost before quantum encoding.
- **VQC Autoencoder** learns digit-specific compression. Anomaly digits (1s) compress
  differently than the trained normal class (0s), producing reconstruction errors.
- **Quantum Distance k-NN** provides geometric anomaly detection in quantum feature space.
- Classical methods benefit from having the full PCA features without the quantum
  encoding bottleneck, but quantum methods demonstrate the feasibility of quantum
  state-based anomaly detection.
- The choice of normal vs anomaly digit significantly affects difficulty. Try changing
  `NORMAL_DIGIT` and `ANOMALY_DIGITS` to explore different setups.